# mnemo Python SDK Demo

**mnemo** is a local-first AI memory layer. It extracts entities and facts from your conversations, builds a persistent knowledge graph, and injects relevant memory into future prompts.

- Works with Ollama (fully offline), OpenAI, Anthropic, or any OpenAI-compatible API
- Sub-50ms memory retrieval
- Simple REST API — this SDK wraps it in a Pythonic interface

> **Prerequisites:** Run `cargo run -p mnemo-api` (or `docker compose up`) to start the mnemo server before running these cells.

## Installation

Install the SDK via pip (uncomment if not already installed):

In [ ]:
# Uncomment to install:
# import sys
# !{sys.executable} -m pip install mnemo-sdk

## Setup

Import the SDK and create a client connected to your local mnemo server.

In [ ]:
from mnemo import MnemoClient, AsyncMnemoClient

# Default connects to http://localhost:8080
client = MnemoClient()
print("Client ready:", client.base_url)

## Health Check

Verify the server is running and inspect its configuration.

In [ ]:
health = client.health()
print(f"Status:       {health.status}")
print(f"Version:      {health.version}")
print(f"DB connected: {health.db_connected}")
print(f"LLM reachable:{health.llm_reachable}")
print(f"Provider:     {health.provider_type} / {health.provider_model}")
print(f"Uptime:       {health.uptime_seconds}s")

## Ingesting Memories

Ingest five realistic memories. mnemo extracts entities (people, tools, concepts) and builds a knowledge graph automatically.

In [ ]:
memories = [
    "I am a software engineer at Acme Corp working on backend infrastructure.",
    "I use Rust and Python daily — Rust for systems code, Python for ML pipelines.",
    "My current project is a vector database called vecdb built in Rust with HNSW indexing.",
    "I prefer dark mode, Neovim for editing, and fish shell for my terminal.",
    "My team at Acme uses GitHub for source control and Linear for project management.",
]

for text in memories:
    result = client.ingest(text, source="notebook-demo")
    print(f"✓ Ingested | entities={result.entities_extracted} relations={result.relations_extracted} time={result.processing_time_ms}ms")

## Retrieving Memory Context

Query the memory store. mnemo returns a formatted context string ready to inject into your LLM prompt.

In [ ]:
result = client.retrieve("what tools does the user prefer?")
print("=== Context Prompt ===")
print(result.context_prompt)
print(f"\n--- Retrieved {len(result.entities)} entities, {len(result.chunks)} chunks ---")

## Listing Entities

Browse all entities mnemo has extracted from your memories. If pandas is installed, display as a DataFrame.

In [ ]:
entities = client.list_entities(limit=20)
print(f"Found {len(entities)} entities\n")

try:
    import pandas as pd
    df = pd.DataFrame([
        {
            "name": e.name,
            "type": e.entity_type,
            "confidence": round(e.confidence, 2),
            "sources": e.source_count,
        }
        for e in entities
    ])
    display(df)  # noqa: F821 — Jupyter built-in
except ImportError:
    for e in entities:
        print(f"  [{e.entity_type:12}] {e.name} (confidence={e.confidence:.2f})")

## Memory Statistics

Inspect the current state of the memory store and knowledge graph.

In [ ]:
stats = client.stats()
print("mnemo memory stats")
print("──────────────────")
print(f"  Entities:    {stats.entity_count}")
print(f"  Chunks:      {stats.chunk_count}")
print(f"  Graph nodes: {stats.node_count}")
print(f"  Graph edges: {stats.edge_count}")
print(f"  Uptime:      {stats.uptime_seconds}s")

## Async Usage

For async applications (FastAPI, LangChain async chains, etc.), use `AsyncMnemoClient`. It mirrors the sync client exactly but all methods are `async def`.

In [ ]:
import asyncio

async def async_demo():
    async with AsyncMnemoClient() as aclient:
        # Ingest asynchronously
        resp = await aclient.ingest(
            "I also maintain an open-source CLI tool called mnemo.",
            source="async-demo",
        )
        print(f"Async ingest: chunk_id={resp.chunk_id}")

        # Retrieve context
        context = await aclient.get_context("what open source work does the user do?")
        print(f"\nAsync context:\n{context}")

# In a Jupyter notebook, use await directly:
# await async_demo()
#
# In a regular script:
asyncio.run(async_demo())

## Integration with LangChain

mnemo can act as a drop-in persistent memory layer for any LangChain-style chain.
The `MnemoMemory` class below implements the save/load interface that most LangChain memory components use,
so you can swap it in without changing your chain code.

In [ ]:
# Use mnemo as a drop-in memory layer for any LLM chain
from mnemo import MnemoClient

class MnemoMemory:
    """Drop-in mnemo memory for LangChain-style usage."""
    def __init__(self, session_id: str = None):
        self.client = MnemoClient()
        self.session_id = session_id

    def save(self, text: str) -> None:
        self.client.ingest(text, session_id=self.session_id)

    def load(self, query: str) -> str:
        return self.client.get_context(query, session_id=self.session_id)

# Example usage
memory = MnemoMemory(session_id="my-chat-session")
memory.save("The user's name is Alex and they work at a fintech startup")
context = memory.load("who is the user?")
print(context)